In [0]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:

# paths
BRONZE_PATH = "/Volumes/dev/electroflow_pipeline/bronze_data/"
SILVER_PATH = "/Volumes/dev/electroflow_pipeline/silver_data/"

# tables
bronze_customers_table = "bronze_customers"
bronze_products_table = "bronze_products"
bronze_orders_table = "bronze_orders"
bronze_payments_table = "bronze_order_payments"
bronze_coupons_table = "bronze_coupons"

silver_customers_table = "silver_customers"
silver_products_table = "silver_products"
silver_orders_table = "silver_orders"
silver_payments_table = "silver_order_payments"
silver_coupons_table = "silver_coupons"

__functions

In [0]:
#customers
def transform_customers():
    print(f"--- cleaning customers...")

    # load data from bronze
    df = spark.read.format("delta").load(BRONZE_PATH + bronze_customers_table)

    # Clean & Refine
    # - Deduplicate by customer_id
    # - Handle mixed date formats (yyyy-MM-dd and MM/dd/yyyy)
    # - Fill missing phone numbers
    df_clean = df.dropDuplicates(["customer_id"])\
        .withColumn("join_date", F.coalesce(
            F.try_to_date("join_date", "yyyy-MM-dd"), 
            F.try_to_date("join_date", "MM/dd/yyyy")
        )) \
        .withColumn("phone_number", F.coalesce(F.col("phone_number"), F.lit("unknown")))\
        .withColumn("gender_int", 
            F.when(F.col("gender") == "male", 1)
             .when(F.col("gender") == "female", 2)
             .otherwise(0))
    print("cleaning done")
    
    #write to silver column
    df_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(SILVER_PATH + silver_customers_table)
    print("writing customers done")
    return df_clean

In [0]:

#orders
def transform_orders():
    print("🛒 Cleaning Orders...")

    #load
    df = spark.read.format("delta").load(BRONZE_PATH+bronze_orders_table)

    #clean and explode orders
    df_clean = df.select("*", F.explode("items").alias("item"))\
        .select(
            "*",
            F.col("item.product_id").alias("product_d"),
            F.col("item.quantity").alias("quantity"),
            F.col("item.unit_price").alias("unit_price"),
            F.col("item.item_total").alias("item_total")
        ).drop("item", "items")
    print("cleaning exploding orders is done done")

    #write to silver column
    df_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(SILVER_PATH + silver_orders_table)
    print("writing orders done")
    return df_clean

In [0]:

#products
def transform_products():
    print("📦 Cleaning Products...")
    
    # Load
    df = spark.read.format("delta").load(BRONZE_PATH+bronze_products_table)
    
    # Clean & Refine
    # - Deduplicate by product_id
    # - Ensure price is a decimal for calculation accuracy
    df_clean = df.dropDuplicates(["product_id"]) \
        .withColumn("price", F.col("base_price").cast("decimal(18,2)"))
    print("cleaning products is done done")

    # Write
    df_clean.write.format("delta").mode("overwrite").save(SILVER_PATH + silver_products_table)
    print("writing products done")
    return df_clean

In [0]:

#---execute the functions
silver_customers = transform_customers()
silver_orders = transform_orders()
silver_produts = transform_products()